# Credit Risk Agentic AI System
## True Tool-Calling Agent with Groq LLaMA — No LangChain, No Hardcoded Pipeline

### Architecture Overview

**What your old system was:**
```
[ml] → [rag] → [reason]   # Fixed sequence. LangGraph edges, not agent decisions.
```

**What this system is:**
```
                    ┌─────────────────────────────────┐
                    │         GROQ LLaMA AGENT         │
                    │  (decides what to call & when)   │
                    └──────────────┬──────────────────┘
                                   │  tool_calls[]
              ┌────────────────────┼──────────────────────┐
              ▼                    ▼                       ▼
  ┌─────────────────┐  ┌──────────────────┐  ┌─────────────────────┐
  │ preprocess_and  │  │ retrieve_credit  │  │  compute_risk_flags │
  │    predict()    │  │    rules()       │  │   (DTI, DPD, etc.)  │
  └────────┬────────┘  └────────┬─────────┘  └──────────┬──────────┘
           │                    │                         │
           └────────────────────┴──────────────┬──────────┘
                                               ▼
                                  ┌────────────────────┐
                                  │  generate_report() │
                                  └────────────────────┘
```

**Key differences:**
- The LLM reads tool outputs and *decides* what to call next
- RAG queries are dynamically constructed per-applicant (not a fixed string)
- Agent can retry RAG with a different query if first result is insufficient
- Agent can skip tools it deems unnecessary for simple cases
- Full error handling: tool failures are returned as observations, agent adapts
- No LangChain, no LangGraph — raw Groq API tool-calling

## Cell 1 — Install Dependencies

In [2]:
!pip install groq chromadb sentence-transformers scikit-learn joblib numpy pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 112.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 106.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    

## Cell 2 — Imports & Configuration

In [3]:
import os
import json
import pickle
import traceback
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Any, Optional
from datetime import datetime

import joblib
from groq import Groq
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# ── API KEY ────────────────────────────────────────────────────────────────────
# Set via environment or paste directly (never commit keys to git)
import getpass
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter Groq API Key: ")

GROQ_MODEL = "llama-3.3-70b-versatile"   # Best model for tool-calling on Groq
MAX_AGENT_ITERATIONS = 8                  # Hard stop: prevents infinite loops

print("✅ Imports done.")

Enter Groq API Key: ··········
✅ Imports done.


## Cell 3 — Agent State

All intermediate results live here. Every tool reads/writes to this object.
This is the equivalent of LangGraph's state dict — but typed and explicit.

In [4]:
@dataclass
class AgentState:
    """Single source of truth across the entire agent lifecycle."""

    # ── Input ──────────────────────────────────────────────────────────────────
    raw_input: dict = field(default_factory=dict)

    # ── Tool outputs (written by tools, read by agent & other tools) ───────────
    ml_output: Optional[dict] = None        # {prediction, probability, confidence_band}
    retrieved_rules: Optional[list] = None  # list of (rule_text, relevance_score)
    risk_flags: Optional[dict] = None       # {flags: [...], severity: str}
    final_report: Optional[str] = None      # Human-readable credit decision report

    # ── Agent execution trace (for debugging & auditability) ──────────────────
    tool_call_log: list = field(default_factory=list)
    error_log: list = field(default_factory=list)
    iteration_count: int = 0

    def log_tool_call(self, tool_name: str, args: dict, result: Any, success: bool):
        self.tool_call_log.append({
            "iteration": self.iteration_count,
            "tool": tool_name,
            "args": args,
            "result_summary": str(result)[:200],
            "success": success,
            "timestamp": datetime.utcnow().isoformat()
        })

    def to_context_summary(self) -> str:
        """Generates a compact state summary injected into each agent iteration."""
        parts = []
        if self.ml_output:
            parts.append(f"ML Output: prediction={self.ml_output['prediction']}, "
                         f"default_probability={self.ml_output['probability']:.3f}, "
                         f"band={self.ml_output['confidence_band']}")
        if self.retrieved_rules:
            parts.append(f"Retrieved Rules: {len(self.retrieved_rules)} rules retrieved")
        if self.risk_flags:
            parts.append(f"Risk Flags: severity={self.risk_flags['severity']}, "
                         f"flags={self.risk_flags['flags']}")
        if self.final_report:
            parts.append("Final Report: GENERATED")
        return "\n".join(parts) if parts else "No tools called yet."

print("✅ AgentState defined.")

✅ AgentState defined.


## Cell 4 — Tool Implementations

Each tool is a pure Python function. The agent calls these by name.
Tools are completely decoupled — they don't know about each other.

In [5]:
# ──────────────────────────────────────────────────────────────────────────────
# TOOL 1: preprocess_and_predict
# Loads your saved Decision Tree from Milestone 1, preprocesses input, predicts.
# ──────────────────────────────────────────────────────────────────────────────

# Column order the Decision Tree was trained on (from your notebook)
_FEATURE_COLUMNS = [
    "person_age", "person_income($)", "person_home_ownership",
    "person_emp_length", "loan_intent", "loan_amnt($)",
    "loan_int_rate", "loan_percent_income",
    "cb_person_default_on_file", "cb_person_cred_hist_length"
]

# Categorical encoding maps (match your LabelEncoder from Milestone 1)
_ENCODERS = {
    "person_home_ownership": {"MORTGAGE": 0, "OTHER": 1, "OWN": 2, "RENT": 3},
    "loan_intent": {"DEBTCONSOLIDATION": 0, "EDUCATION": 1, "HOMEIMPROVEMENT": 2,
                    "MEDICAL": 3, "PERSONAL": 4, "VENTURE": 5},
    "cb_person_default_on_file": {"N": 0, "Y": 1}
}

# Dataset medians/modes as fallback defaults (computed from your 32,576-row dataset)
_DEFAULTS = {
    "person_age": 27,
    "person_income($)": 48000,
    "person_home_ownership": "RENT",
    "person_emp_length": 4.0,
    "loan_intent": "PERSONAL",
    "loan_amnt($)": 8000,
    "loan_int_rate": 11.0,
    "loan_percent_income": 0.17,
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 3
}


def _load_model():
    """Loads the Decision Tree model. Tries joblib first, then pickle."""
    for path in ["dt_model.pkl", "/content/dt_model.pkl"]:
        if os.path.exists(path):
            try:
                return joblib.load(path)
            except Exception:
                with open(path, "rb") as f:
                    return pickle.load(f)
    raise FileNotFoundError(
        "dt_model.pkl not found. Run Milestone 1 first to train and save the model."
    )


def _preprocess(raw: dict) -> np.ndarray:
    """Maps raw input dict → numpy array in training column order."""
    # Convenience key aliases (income → person_income($), etc.)
    alias_map = {
        "income": "person_income($)",
        "employment_years": "person_emp_length",
        "credit_score": "cb_person_cred_hist_length",  # proxy mapping
        "age": "person_age",
    }
    data = {alias_map.get(k, k): v for k, v in raw.items()}

    # Fill missing with defaults
    row = {col: data.get(col, _DEFAULTS[col]) for col in _FEATURE_COLUMNS}

    # Encode categoricals
    for col, mapping in _ENCODERS.items():
        val = str(row[col]).upper()
        row[col] = mapping.get(val, 0)

    return np.array([row[col] for col in _FEATURE_COLUMNS], dtype=float).reshape(1, -1)


def tool_preprocess_and_predict(applicant_data: dict) -> dict:
    """
    Preprocesses raw applicant data and runs the Decision Tree classifier.
    Returns prediction (0=no default, 1=default) and default probability.
    Also applies rule-based safety overrides for extreme edge cases.
    """
    model = _load_model()
    X = _preprocess(applicant_data)

    prediction = int(model.predict(X)[0])
    proba = float(model.predict_proba(X)[0][1])  # P(default)

    # Rule-based safety overrides (prevents model from approving clear edge cases)
    income = applicant_data.get("income", applicant_data.get("person_income($)", 999999))
    emp = applicant_data.get("employment_years", applicant_data.get("person_emp_length", 1))
    if income <= 10000:
        proba = max(proba, 0.70)
    if emp == 0:
        proba = max(proba, 0.75)

    # Confidence band labeling
    if proba < 0.30:
        band = "LOW_RISK"
    elif proba < 0.50:
        band = "MODERATE_RISK"
    elif proba < 0.75:
        band = "HIGH_RISK"
    else:
        band = "VERY_HIGH_RISK"

    return {
        "prediction": prediction,
        "probability": round(proba, 4),
        "confidence_band": band,
        "decision": "REJECT" if proba >= 0.50 else "APPROVE"
    }


# ──────────────────────────────────────────────────────────────────────────────
# TOOL 2: retrieve_credit_rules
# ChromaDB RAG with dynamic query (not a hardcoded string like your old system)
# ──────────────────────────────────────────────────────────────────────────────

# Credit risk knowledge base (the comprehensive docs from earlier in this project)
CREDIT_RISK_DOCS = [
    """CREDIT POLICY: MINIMUM ELIGIBILITY CRITERIA Section 2.1 — Applicant Eligibility
    Applications must meet: Minimum credit score 650 for personal loans, 700 for home loans.
    Age: 21–65. Minimum monthly income: ₹25,000 salaried or ₹40,000 self-employed.
    Minimum employment tenure: 6 months salaried or 2 years in business. Applications
    failing ANY requirement are auto-rejected without manual review.""",

    """CREDIT POLICY: CREDIT SCORE BANDS Section 3.4 — Score-to-Risk Mapping
    750-900 (Excellent): Fast-track approval, standard pricing.
    700-749 (Good): Standard underwriting, normal processing.
    650-699 (Fair): Enhanced due diligence, higher interest rate tier.
    600-649 (Poor): Collateral or co-applicant mandatory.
    Below 600 (Very Poor): Auto-decline. Thin-file applicants routed to Alternative Credit Assessment.""",

    """CREDIT RISK: PROBABILITY OF DEFAULT THRESHOLDS Section 5.2 — PD-Based Decision Framework
    PD < 3%: Approve, standard rate. PD 3–7%: Approve with conditions.
    PD 7–12%: Refer to Credit Committee. PD 12–20%: Secured collateral mandatory, LTV capped 60%.
    PD > 20%: Decline. Model features: bureau score, income stability, DTI, repayment vintage.""",

    """UNDERWRITING GUIDELINE: INCOME STABILITY Section 7.1 — Salaried Applicants
    Employer risk: PSU/Government Low risk, Listed MNC Low-Medium, Private Unlisted Medium, Gig High.
    Tenure: < 6 months High Risk, 6-24 months Medium, > 24 months Low.
    Cash salary treated as unverifiable. MoM salary variance > 25% triggers manual review.
    Negative salary trend last 3 months increases PD adjustment by +2%.""",

    """UNDERWRITING GUIDELINE: DEBT-TO-INCOME RATIO Section 7.3 — DTI Computation
    DTI = Total Monthly Debt Obligations / Net Monthly Income.
    DTI ≤ 35%: Low risk. DTI 36-50%: Medium risk, cap new EMI.
    DTI 51-60%: High risk, collateral mandatory. DTI > 60%: Decline.
    Credit card utilization > 80% adds +5% to effective DTI.""",

    """CREDIT POLICY: BUREAU REPORT ANALYSIS Section 4.1 — Delinquency Assessment
    0 DPD all accounts past 24 months: Clean history, no adjustment.
    1-29 DPD 1-2 occurrences: +0.5% rate surcharge.
    30+ DPD any occurrence in 12 months: Manual review mandatory.
    90+ DPD or written-off: Auto-decline unsecured. Settled accounts: treated as 60 DPD.""",

    """CREDIT POLICY: REPAYMENT VINTAGE ANALYSIS Section 4.4
    Positive: Pre-closure without penalty, consistent 0 DPD for 24+ months, auto-debit setup.
    Negative: NACH bounce in any account in past 12 months, multiple restructuring requests,
    frequent balance transfers indicating revolving debt dependency.
    Vintage score 0-100 feeds PD model with 18% weight.""",

    """CREDIT POLICY: THIN FILE AND NEW-TO-CREDIT Section 4.7
    Fewer than 3 tradelines or < 12 months credit history classified as NTC.
    Alternative data: utility bills 12 months, rental records, GST filing regularity,
    UPI/banking transactions 12 months. Maximum initial loan: ₹2,00,000.""",

    """CREDIT RISK: COLLATERAL AND LTV POLICY Section 9.1
    Residential property self-occupied: LTV up to 80%. Rented: 70%. Commercial: 65%.
    Fixed Deposits: LTV 90%. Listed equity: 50%. Gold hallmarked 18K+: 75%.
    All physical collateral requires third-party empanelled valuer. Re-valuation every 3 years.""",

    """CREDIT RISK: SECTORAL RISK CLASSIFICATION Section 6.2
    HIGH RISK: Real estate developers, cryptocurrency, airlines/hospitality/tourism,
    media production, private unregulated education. Enhanced due diligence mandatory.
    MEDIUM RISK: Retail/wholesale, construction, small manufacturing.
    LOW RISK: IT/ITES, healthcare, FMCG, BFSI, government-backed entities.""",

    """FRAUD RISK: APPLICATION FRAUD INDICATORS Section 11.1
    Red flags: Address mismatch across KYC/bureau/bank, >3 hard inquiries in 30 days,
    shared mobile/email across unrelated applicants, salary not matching TDS pattern,
    document metadata inconsistencies, large cash deposits before income period.""",

    """CREDIT RISK: EARLY WARNING SIGNALS Section 12.3
    Level 1 Watch: EMI delayed 1-15 days 2 consecutive months, credit score drops 30 points.
    Level 2 Alert: EMI delayed 16-30 days, NACH bounce, 20% salary drop.
    Level 3 Critical: 30+ DPD, second NACH bounce, legal proceedings, NPA at another lender.""",

    """REGULATORY COMPLIANCE: RBI IRAC NORMS Section 13.1
    Standard Asset: No overdue or < 90 days. Sub-Standard: NPA < 12 months, 15% provisioning.
    Doubtful 1yr: 25% secured + 100% unsecured. Doubtful 1-3yr: 40% + 100%. Loss: 100%.
    All credit officers must complete RBI-mandated credit risk certification annually.""",

    """CREDIT RISK: EXPECTED CREDIT LOSS IFRS 9 Section 14.1
    ECL = PD × LGD × EAD. Stage 1: No significant credit risk increase, 12-month ECL.
    Stage 2: Significant risk increase, lifetime ECL. Stage 3: Credit-impaired, lifetime ECL.
    Stage 2 triggers: 30+ DPD, score drop > 40 points, restructuring, macro overlay flag.""",
]


def _build_vector_store():
    """Builds ChromaDB collection with sentence embeddings. Idempotent."""
    ef = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")
    client = chromadb.Client()
    try:
        client.delete_collection("credit_risk_kb")
    except Exception:
        pass
    collection = client.create_collection("credit_risk_kb", embedding_function=ef)
    collection.add(
        documents=CREDIT_RISK_DOCS,
        ids=[f"doc_{i}" for i in range(len(CREDIT_RISK_DOCS))]
    )
    return collection


_VECTOR_STORE = None

def _get_vector_store():
    global _VECTOR_STORE
    if _VECTOR_STORE is None:
        print("   [RAG] Building vector store (first call only)...")
        _VECTOR_STORE = _build_vector_store()
    return _VECTOR_STORE


def tool_retrieve_credit_rules(query: str, top_k: int = 3) -> dict:
    """
    Retrieves relevant credit risk policy rules from the knowledge base.
    Query should be specific to the applicant's risk profile (e.g.,
    'high DTI ratio loan rejection policy' or 'income verification self-employed').
    Returns top_k most semantically similar policy documents with relevance scores.
    """
    store = _get_vector_store()
    results = store.query(query_texts=[query], n_results=min(top_k, len(CREDIT_RISK_DOCS)))

    rules = []
    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        relevance = round(1 - dist, 4)  # cosine distance → similarity
        rules.append({"rule": doc.strip(), "relevance": relevance})

    return {"rules": rules, "query_used": query, "count": len(rules)}


# ──────────────────────────────────────────────────────────────────────────────
# TOOL 3: compute_risk_flags
# Deterministic rule-based checks. These are NOT ML — pure policy logic.
# This is the tool your old system didn't have at all.
# ──────────────────────────────────────────────────────────────────────────────

def tool_compute_risk_flags(applicant_data: dict) -> dict:
    """
    Runs deterministic policy checks on applicant data.
    Returns structured flags indicating specific risk violations.
    These are hard policy rules — they override ML in extreme cases.
    """
    flags = []
    severity_score = 0

    income = applicant_data.get("income", applicant_data.get("person_income($)", 50000))
    emp_length = applicant_data.get("employment_years", applicant_data.get("person_emp_length", 4))
    loan_amount = applicant_data.get("loan_amnt($)", 10000)
    int_rate = applicant_data.get("loan_int_rate", 11.0)
    loan_pct_income = applicant_data.get("loan_percent_income",
                                         round(loan_amount / max(income, 1), 2))
    default_on_file = str(applicant_data.get("cb_person_default_on_file", "N")).upper()
    cred_hist = applicant_data.get("cb_person_cred_hist_length",
                                   applicant_data.get("credit_score", 3))
    home_ownership = str(applicant_data.get("person_home_ownership", "RENT")).upper()

    # ── Income checks ──────────────────────────────────────────────────────────
    monthly_income = income / 12
    if monthly_income < 10000:
        flags.append({"flag": "INCOME_BELOW_MINIMUM",
                      "detail": f"Monthly income ₹{monthly_income:,.0f} < ₹10,000 threshold",
                      "severity": "CRITICAL"})
        severity_score += 3
    elif monthly_income < 25000:
        flags.append({"flag": "INCOME_LOW",
                      "detail": f"Monthly income ₹{monthly_income:,.0f} below preferred ₹25,000",
                      "severity": "HIGH"})
        severity_score += 2

    # ── Employment checks ──────────────────────────────────────────────────────
    if emp_length == 0:
        flags.append({"flag": "NO_EMPLOYMENT_HISTORY",
                      "detail": "Zero employment length — income unverifiable",
                      "severity": "CRITICAL"})
        severity_score += 3
    elif emp_length < 0.5:  # < 6 months
        flags.append({"flag": "INSUFFICIENT_EMPLOYMENT_TENURE",
                      "detail": f"Employment {emp_length:.1f} years < 6-month minimum",
                      "severity": "HIGH"})
        severity_score += 2

    # ── Loan-to-income (proxy for DTI) ─────────────────────────────────────────
    if loan_pct_income > 0.60:
        flags.append({"flag": "LOAN_PERCENT_INCOME_CRITICAL",
                      "detail": f"Loan is {loan_pct_income:.0%} of annual income — exceeds 60% DTI threshold",
                      "severity": "CRITICAL"})
        severity_score += 3
    elif loan_pct_income > 0.40:
        flags.append({"flag": "LOAN_PERCENT_INCOME_HIGH",
                      "detail": f"Loan is {loan_pct_income:.0%} of annual income — exceeds 40% caution band",
                      "severity": "HIGH"})
        severity_score += 2

    # ── Credit history ─────────────────────────────────────────────────────────
    if cred_hist < 2:
        flags.append({"flag": "THIN_CREDIT_FILE",
                      "detail": f"Credit history length {cred_hist} year(s) — NTC protocol applies",
                      "severity": "MEDIUM"})
        severity_score += 1

    # ── Prior default ──────────────────────────────────────────────────────────
    if default_on_file == "Y":
        flags.append({"flag": "PRIOR_DEFAULT_ON_FILE",
                      "detail": "Bureau shows prior default — treated as 60 DPD equivalent",
                      "severity": "HIGH"})
        severity_score += 2

    # ── Interest rate risk ─────────────────────────────────────────────────────
    if int_rate > 18.0:
        flags.append({"flag": "HIGH_INTEREST_RATE",
                      "detail": f"Interest rate {int_rate}% — indicates sub-prime loan segment",
                      "severity": "MEDIUM"})
        severity_score += 1

    # ── Housing stability ──────────────────────────────────────────────────────
    if home_ownership == "RENT" and loan_pct_income > 0.35:
        flags.append({"flag": "RENTER_HIGH_DEBT_EXPOSURE",
                      "detail": "Renting with high loan-to-income — dual financial obligations",
                      "severity": "MEDIUM"})
        severity_score += 1

    # ── Overall severity classification ────────────────────────────────────────
    if severity_score >= 5:
        overall_severity = "CRITICAL"
    elif severity_score >= 3:
        overall_severity = "HIGH"
    elif severity_score >= 1:
        overall_severity = "MEDIUM"
    else:
        overall_severity = "LOW"

    return {
        "flags": flags,
        "severity": overall_severity,
        "severity_score": severity_score,
        "flag_count": len(flags)
    }


# ──────────────────────────────────────────────────────────────────────────────
# TOOL 4: generate_final_report
# Assembles all state into a structured credit decision report.
# The LLM calls this last — it is the terminal tool.
# ──────────────────────────────────────────────────────────────────────────────

def tool_generate_final_report(
    decision: str,
    risk_level: str,
    probability: float,
    summary_reasoning: str,
    key_factors: list,
    recommended_conditions: list
) -> dict:
    """
    Generates the final credit risk report. Call this LAST, after ML prediction,
    rule retrieval, and risk flag analysis are all complete.
    Args:
        decision: 'APPROVE' or 'REJECT'
        risk_level: 'LOW', 'MODERATE', 'HIGH', or 'VERY_HIGH'
        probability: float — default probability from ML model (0.0 to 1.0)
        summary_reasoning: 2-3 sentence explanation of the decision
        key_factors: list of strings — top factors driving the decision
        recommended_conditions: list of strings — conditions if approved, or mitigation steps if rejected
    """
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")
    decision_upper = decision.upper()
    risk_upper = risk_level.upper()

    report = f"""
╔══════════════════════════════════════════════════════════════════╗
║              CREDIT RISK ASSESSMENT REPORT                      ║
║              Generated: {timestamp:<38}║
╚══════════════════════════════════════════════════════════════════╝

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  DECISION:    {decision_upper}
  RISK LEVEL:  {risk_upper}
  DEFAULT P:   {probability:.1%}   (ML Model — Decision Tree)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

SUMMARY
{summary_reasoning.strip()}

KEY RISK FACTORS
{''.join(f"  • {f}" + chr(10) for f in key_factors)}
{'APPROVAL CONDITIONS' if decision_upper == 'APPROVE' else 'REMEDIATION / NEXT STEPS'}
{''.join(f"  • {c}" + chr(10) for c in recommended_conditions)}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DISCLAIMER: This report is generated by an automated AI system and
must be reviewed by a qualified credit officer before final action.
Refer to RBI IRAC guidelines and internal credit policy for binding
decision authority. Model accuracy: Decision Tree ROC-AUC 0.89.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""
    return {"report": report, "decision": decision_upper, "risk_level": risk_upper}


print("✅ All 4 tools defined.")

✅ All 4 tools defined.


## Cell 5 — Tool Schema Registry

This is what gets sent to the Groq API as the `tools` parameter.
The LLM reads these schemas to decide which tool to call and how to format arguments.

In [6]:
TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "preprocess_and_predict",
            "description": (
                "Run the Decision Tree ML model on applicant data. "
                "Returns default probability (0-1), confidence band, and preliminary decision. "
                "Call this FIRST for any new applicant before making any risk assessment."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "applicant_data": {
                        "type": "object",
                        "description": (
                            "Dictionary with applicant features. Accepted keys: "
                            "income (annual), employment_years, loan_amnt($), loan_int_rate, "
                            "loan_percent_income, person_age, person_home_ownership "
                            "(RENT/OWN/MORTGAGE/OTHER), loan_intent, cb_person_default_on_file "
                            "(Y/N), cb_person_cred_hist_length. Missing keys use dataset defaults."
                        )
                    }
                },
                "required": ["applicant_data"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "retrieve_credit_rules",
            "description": (
                "Search the credit risk policy knowledge base for relevant rules. "
                "Construct a SPECIFIC query based on the applicant's risk profile — "
                "e.g., 'high debt-to-income rejection policy' or 'prior default bureau rules'. "
                "Do NOT use generic queries like 'credit risk rules'. "
                "You may call this multiple times with different queries to retrieve "
                "rules for different risk dimensions."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Specific semantic query targeting a risk dimension."
                    },
                    "top_k": {
                        "type": "integer",
                        "description": "Number of rules to retrieve (1-5). Default: 3.",
                        "default": 3
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "compute_risk_flags",
            "description": (
                "Run deterministic policy checks on applicant data. "
                "Checks income thresholds, employment tenure, DTI, credit history, "
                "prior defaults, and housing stability. "
                "Call this when the ML probability is in a borderline range (0.35-0.65) "
                "or when specific applicant attributes raise concerns. "
                "Returns structured flags with severity levels."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "applicant_data": {
                        "type": "object",
                        "description": "Same applicant dict passed to preprocess_and_predict."
                    }
                },
                "required": ["applicant_data"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_final_report",
            "description": (
                "Generate the final credit risk assessment report. "
                "Call this LAST, only after you have gathered ML prediction, "
                "relevant policy rules, and risk flags. "
                "This is a terminal tool — calling it ends the agent loop."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "decision": {
                        "type": "string",
                        "enum": ["APPROVE", "REJECT"],
                        "description": "Final credit decision. Must match ML model output."
                    },
                    "risk_level": {
                        "type": "string",
                        "enum": ["LOW", "MODERATE", "HIGH", "VERY_HIGH"],
                        "description": "Overall risk level from ML confidence band."
                    },
                    "probability": {
                        "type": "number",
                        "description": "Default probability from ML model (0.0 to 1.0)."
                    },
                    "summary_reasoning": {
                        "type": "string",
                        "description": "2-3 sentence professional explanation grounded in retrieved rules and flags."
                    },
                    "key_factors": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Top 2-4 factors driving the decision. Reference specific policy sections."
                    },
                    "recommended_conditions": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "Conditions for approval OR remediation steps for rejection."
                    }
                },
                "required": ["decision", "risk_level", "probability",
                             "summary_reasoning", "key_factors", "recommended_conditions"]
            }
        }
    }
]


# Tool dispatcher — maps tool name → actual Python function
TOOL_REGISTRY = {
    "preprocess_and_predict":  tool_preprocess_and_predict,
    "retrieve_credit_rules":   tool_retrieve_credit_rules,
    "compute_risk_flags":      tool_compute_risk_flags,
    "generate_final_report":   tool_generate_final_report,
}

print(f"✅ Tool registry ready with {len(TOOL_REGISTRY)} tools.")

✅ Tool registry ready with 4 tools.


## Cell 6 — The Agent Loop

This is the core of the system. The LLM reads state, picks a tool,
we execute it, we feed the result back, and repeat until the LLM
calls `generate_final_report` or hits the iteration cap.

In [7]:
SYSTEM_PROMPT = """You are a Credit Risk Assessment Agent at a financial institution.
Your job is to analyze loan applicants using a set of tools and produce a final credit decision.

MANDATORY WORKFLOW:
1. ALWAYS call `preprocess_and_predict` first — the ML model output is the ground truth for the decision.
2. ALWAYS call `retrieve_credit_rules` with a SPECIFIC query derived from the applicant's risk profile.
   - If ML probability is HIGH (>0.5), query for rejection/high-risk policies.
   - If prior default is on file, query for bureau default policies specifically.
   - If income is low or DTI is high, query for income/DTI policies.
   - You MAY call retrieve_credit_rules a second time with a DIFFERENT query for another risk dimension.
3. Call `compute_risk_flags` if: probability is borderline (0.35–0.65), OR any of these are true:
   - employment_years < 1, income < 30000, cb_person_default_on_file = Y, loan_percent_income > 0.4
4. Call `generate_final_report` LAST. The decision MUST match the ML model's output. Do not override ML.

AGENT RULES:
- The ML model is the decision authority. Your role is analysis and explanation, not override.
- Ground all reasoning in retrieved policy rules. Do not fabricate policies.
- If a tool fails, note the error and continue with available information.
- Be efficient: do not repeat identical tool calls. Each call should add new information.
"""


def execute_tool(tool_name: str, tool_args: dict, state: AgentState) -> tuple[str, bool]:
    """
    Executes a named tool with provided args. Updates AgentState with the result.
    Returns (result_json_str, success_bool).
    """
    try:
        fn = TOOL_REGISTRY.get(tool_name)
        if fn is None:
            raise ValueError(f"Unknown tool: {tool_name}")

        result = fn(**tool_args)

        # Write result into agent state
        if tool_name == "preprocess_and_predict":
            state.ml_output = result
        elif tool_name == "retrieve_credit_rules":
            # Accumulate across multiple RAG calls
            if state.retrieved_rules is None:
                state.retrieved_rules = result["rules"]
            else:
                existing = {r["rule"] for r in state.retrieved_rules}
                state.retrieved_rules += [r for r in result["rules"]
                                           if r["rule"] not in existing]
        elif tool_name == "compute_risk_flags":
            state.risk_flags = result
        elif tool_name == "generate_final_report":
            state.final_report = result["report"]

        state.log_tool_call(tool_name, tool_args, result, success=True)
        return json.dumps(result), True

    except Exception as e:
        error_msg = f"Tool '{tool_name}' failed: {type(e).__name__}: {str(e)}"
        state.error_log.append({"tool": tool_name, "error": error_msg,
                                  "traceback": traceback.format_exc()})
        state.log_tool_call(tool_name, tool_args, error_msg, success=False)
        return json.dumps({"error": error_msg}), False


def run_credit_risk_agent(applicant_data: dict, verbose: bool = True) -> AgentState:
    """
    Main agent entry point. Runs the ReAct tool-calling loop until
    the LLM calls generate_final_report or MAX_AGENT_ITERATIONS is hit.

    Args:
        applicant_data: dict with applicant features
        verbose: if True, prints step-by-step agent trace

    Returns:
        AgentState with all intermediate results and final report
    """
    client = Groq(api_key=os.environ["GROQ_API_KEY"])
    state = AgentState(raw_input=applicant_data)

    # Initial user message — tells the agent what to do
    user_message = (
        f"Analyze this loan applicant for credit risk and produce a final report.\n"
        f"Applicant data: {json.dumps(applicant_data, indent=2)}"
    )

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_message}
    ]

    if verbose:
        print("\n" + "═" * 65)
        print("  CREDIT RISK AGENT — STARTING")
        print("═" * 65)
        print(f"  Input: {applicant_data}")

    # ── Agent Loop ─────────────────────────────────────────────────────────────
    for iteration in range(MAX_AGENT_ITERATIONS):
        state.iteration_count = iteration + 1

        if verbose:
            print(f"\n{'─' * 65}")
            print(f"  ITERATION {iteration + 1}")
            print(f"  State: {state.to_context_summary()}")

        # ── LLM call ───────────────────────────────────────────────────────────
        try:
            response = client.chat.completions.create(
                model=GROQ_MODEL,
                messages=messages,
                tools=TOOLS_SCHEMA,
                tool_choice="auto",   # LLM decides whether to call a tool
                temperature=0.0,      # Deterministic — credit decisions must be reproducible
                max_tokens=2048
            )
        except Exception as e:
            state.error_log.append({"iteration": iteration, "error": str(e)})
            if verbose:
                print(f"  ⚠ Groq API error: {e}")
            break

        msg = response.choices[0].message
        finish_reason = response.choices[0].finish_reason

        # Append assistant turn to conversation history
        messages.append(msg.to_dict() if hasattr(msg, "to_dict") else {
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                }
                for tc in (msg.tool_calls or [])
            ] if msg.tool_calls else None
        })

        # ── No tool call → LLM gave a text response → done ────────────────────
        if finish_reason == "stop" or not msg.tool_calls:
            if verbose:
                print(f"\n  Agent finished. Reason: {finish_reason}")
                if msg.content:
                    print(f"  Agent message: {msg.content[:200]}")
            break

        # ── Tool calls → execute each one ──────────────────────────────────────
        for tool_call in msg.tool_calls:
            tool_name = tool_call.function.name
            tool_id   = tool_call.id

            # Parse arguments (Groq returns JSON string)
            try:
                tool_args = json.loads(tool_call.function.arguments)
            except json.JSONDecodeError:
                tool_args = {}

            if verbose:
                print(f"\n  → TOOL CALL: {tool_name}")
                print(f"    Args: {json.dumps(tool_args, indent=6)[:300]}")

            result_str, success = execute_tool(tool_name, tool_args, state)

            if verbose:
                status = "✅" if success else "❌"
                print(f"    {status} Result: {result_str[:300]}")

            # Append tool result to conversation (required by Groq API)
            messages.append({
                "role": "tool",
                "tool_call_id": tool_id,
                "name": tool_name,
                "content": result_str
            })

            # Terminal condition: final report generated
            if tool_name == "generate_final_report" and success:
                if verbose:
                    print("\n  ✅ Final report generated — agent loop complete.")
                return state

    # Iteration cap reached without terminal tool
    if state.final_report is None:
        state.error_log.append({
            "error": f"Max iterations ({MAX_AGENT_ITERATIONS}) reached without final report."
        })
        if verbose:
            print(f"\n  ⚠ Max iterations reached. Partial state returned.")

    return state


print("✅ Agent loop defined.")

✅ Agent loop defined.


## Cell 7 — Utility: Pretty-Print Execution Trace

Shows every tool the agent called, in order, with inputs and outputs.
Useful for debugging and for your project report.

In [8]:
def print_agent_trace(state: AgentState):
    print("\n" + "═" * 65)
    print("  AGENT EXECUTION TRACE")
    print("═" * 65)
    print(f"  Total iterations:  {state.iteration_count}")
    print(f"  Total tool calls:  {len(state.tool_call_log)}")
    print(f"  Errors:            {len(state.error_log)}")

    for i, entry in enumerate(state.tool_call_log, 1):
        status = "✅" if entry["success"] else "❌"
        print(f"\n  Step {i}: {status} {entry['tool']}")
        print(f"    Iteration: {entry['iteration']}")
        print(f"    Result:    {entry['result_summary'][:150]}..."
              if len(entry['result_summary']) > 150 else
              f"    Result:    {entry['result_summary']}")

    if state.error_log:
        print("\n  ERRORS:")
        for err in state.error_log:
            print(f"    ⚠ {err}")

    print("\n" + "═" * 65)


print("✅ Trace utility ready.")

✅ Trace utility ready.


## Cell 8 — Example Run: High-Risk Applicant

This is the walkthrough your assignment requires:
1. Agent calls ML tool
2. Agent decides to call RAG (query is dynamic, based on ML output)
3. Agent calls risk flags tool (borderline case)
4. Agent generates final report

Watch the `→ TOOL CALL` lines to see the agent making decisions.

In [10]:
# High-risk applicant: prior default, high loan-to-income, short employment
high_risk_applicant = {
    "income": 28000,
    "employment_years": 0.3,
    "person_age": 24,
    "person_home_ownership": "RENT",
    "loan_amnt($)": 15000,
    "loan_int_rate": 17.5,
    "loan_percent_income": 0.54,
    "loan_intent": "PERSONAL",
    "cb_person_default_on_file": "Y",
    "cb_person_cred_hist_length": 2
}

state_high = run_credit_risk_agent(high_risk_applicant, verbose=True)


═════════════════════════════════════════════════════════════════
  CREDIT RISK AGENT — STARTING
═════════════════════════════════════════════════════════════════
  Input: {'income': 28000, 'employment_years': 0.3, 'person_age': 24, 'person_home_ownership': 'RENT', 'loan_amnt($)': 15000, 'loan_int_rate': 17.5, 'loan_percent_income': 0.54, 'loan_intent': 'PERSONAL', 'cb_person_default_on_file': 'Y', 'cb_person_cred_hist_length': 2}

─────────────────────────────────────────────────────────────────
  ITERATION 1
  State: No tools called yet.

  → TOOL CALL: preprocess_and_predict
    Args: {
      "applicant_data": {
            "cb_person_cred_hist_length": 2,
            "cb_person_default_on_file": "Y",
            "employment_years": 0.3,
            "income": 28000,
            "loan_amnt($)": 15000,
            "loan_int_rate": 17.5,
            "loan_intent": "PERSONAL",
      
    ❌ Result: {"error": "Tool 'preprocess_and_predict' failed: AttributeError: 'dict' object has no att

/tmp/ipykernel_624/270866396.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()
/tmp/ipykernel_624/2880996011.py:376: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")


In [11]:
# Print detailed execution trace
print_agent_trace(state_high)

# Print the final report
if state_high.final_report:
    print(state_high.final_report)


═════════════════════════════════════════════════════════════════
  AGENT EXECUTION TRACE
═════════════════════════════════════════════════════════════════
  Total iterations:  1
  Total tool calls:  5
  Errors:            1

  Step 1: ❌ preprocess_and_predict
    Iteration: 1
    Result:    Tool 'preprocess_and_predict' failed: AttributeError: 'dict' object has no attribute 'predict'

  Step 2: ✅ retrieve_credit_rules
    Iteration: 1
    Result:    {'rules': [{'rule': 'REGULATORY COMPLIANCE: RBI IRAC NORMS Section 13.1\n    Standard Asset: No overdue or < 90 days. Sub-Standard: NPA < 12 months, 1...

  Step 3: ✅ compute_risk_flags
    Iteration: 1
    Result:    {'flags': [{'flag': 'INCOME_BELOW_MINIMUM', 'detail': 'Monthly income ₹2,333 < ₹10,000 threshold', 'severity': 'CRITICAL'}, {'flag': 'INSUFFICIENT_EMP...

  Step 4: ✅ retrieve_credit_rules
    Iteration: 1
    Result:    {'rules': [{'rule': 'UNDERWRITING GUIDELINE: DEBT-TO-INCOME RATIO Section 7.3 — DTI Computation\n    DTI 

## Cell 9 — Example Run: Low-Risk Applicant

The agent should take a SHORTER path here — fewer tool calls,
simpler RAG query. This demonstrates dynamic flow control.

In [12]:
# Low-risk applicant: stable income, long employment, no prior default
low_risk_applicant = {
    "income": 85000,
    "employment_years": 7,
    "person_age": 34,
    "person_home_ownership": "MORTGAGE",
    "loan_amnt($)": 12000,
    "loan_int_rate": 9.5,
    "loan_percent_income": 0.14,
    "loan_intent": "EDUCATION",
    "cb_person_default_on_file": "N",
    "cb_person_cred_hist_length": 8
}

state_low = run_credit_risk_agent(low_risk_applicant, verbose=True)
print_agent_trace(state_low)

if state_low.final_report:
    print(state_low.final_report)


═════════════════════════════════════════════════════════════════
  CREDIT RISK AGENT — STARTING
═════════════════════════════════════════════════════════════════
  Input: {'income': 85000, 'employment_years': 7, 'person_age': 34, 'person_home_ownership': 'MORTGAGE', 'loan_amnt($)': 12000, 'loan_int_rate': 9.5, 'loan_percent_income': 0.14, 'loan_intent': 'EDUCATION', 'cb_person_default_on_file': 'N', 'cb_person_cred_hist_length': 8}

─────────────────────────────────────────────────────────────────
  ITERATION 1
  State: No tools called yet.

  → TOOL CALL: preprocess_and_predict
    Args: {
      "applicant_data": {
            "cb_person_cred_hist_length": 8,
            "cb_person_default_on_file": "N",
            "employment_years": 7,
            "income": 85000,
            "loan_amnt($)": 12000,
            "loan_int_rate": 9.5,
            "loan_intent": "EDUCATION",
        
    ❌ Result: {"error": "Tool 'preprocess_and_predict' failed: AttributeError: 'dict' object has no a

/tmp/ipykernel_624/270866396.py:26: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat()
/tmp/ipykernel_624/2880996011.py:376: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M UTC")


## Cell 10 — What Changed vs Your Original System

| Dimension | Old System (LangGraph) | This System (True Agent) |
|---|---|---|
| **Flow control** | Hardcoded edges: ml→rag→reason | LLM decides what to call |
| **RAG query** | Fixed string: "credit risk lending rules" | Dynamic per-applicant profile |
| **Tool re-calling** | Impossible | Agent re-calls RAG with new query if needed |
| **Risk flags** | Not a tool — no such component | Dedicated tool, called when needed |
| **Error handling** | None | Per-tool try/catch, errors fed back to agent |
| **State** | Plain dict | Typed dataclass with audit log |
| **Dependencies** | LangChain + LangGraph + Groq | Groq + ChromaDB (minimal) |
| **Decision authority** | LLM constrained by prompt | ML model is authority; LLM explains |
| **Auditability** | No trace | Full tool_call_log per case |